In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from scipy.ndimage import gaussian_filter

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [6]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))
files

array(['/home/ulyanov/data/solo/phi/synoptic_maps/2279.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2280.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2281.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2282.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2283.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2284.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2285.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2286.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2287.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2288.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2289.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2290.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2291.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2292.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2293.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2294.npz',
       '/home/ulyanov/da

In [8]:
flux_density = []
dates = []

for file in files:
    s = np.load(file, allow_pickle=True)

    mean = s['mean']
    start = s['start']

    #flux_density.append(mean)
    #dates.append(datetime.fromisoformat(start))


In [11]:
start

array(datetime.datetime(2025, 12, 28, 3, 20, 9), dtype=object)

In [2]:
s = np.load('fluxes_car.npz')

flux_density = s['flux_density']
lati = s['lati'].astype(float)
dates = np.array([datetime.fromisoformat(date) for date in s['date']])

dlat = lati[1] - lati[0]
lat = (lati[1:] + lati[:-1]) / 2

In [3]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux_density[4])
plt.ylim(-10,10)
plt.tight_layout()

In [4]:
R = 695e8
u = 1300
d = 500e10
lam1 = 40
lam2 = 70


xi = lati * np.pi / 180 * R
vi = u * flow(lati)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux_density[1].copy()
y[np.abs(lat) > lam2] = 1

Q = []
for t in np.arange(dates[1], dates[23], timedelta(days=1)):
    dt = timedelta(days=1).total_seconds()

    y = advect(y, xi, vi, dt, li)
    y = diffuse(y, xi, d, dt, li)

    y_ = interp1d(flux_density, dates, t)

    y[np.abs(lat) < lam1] = y_[np.abs(lat) < lam1]

    Q.append(y)

Q = np.array(Q)


In [5]:
dates_ = np.arange(dates[1], dates[23], timedelta(days=1))
flux_density_ = interp1d(flux_density, dates, dates_)

plt.figure(figsize=(10,8))
#plt.plot(lat, flux_density_[0])
plt.plot(lat, flux_density_[-1])
plt.plot(lat, Q[-1])
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)
plt.tight_layout()

In [7]:
plt.figure(figsize=(10,8))
plt.imshow(flux_density_.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)#, interpolation='bicubic')
plt.tight_layout()

In [46]:
plt.figure(figsize=(10,8))
plt.plot(lati, vi)